In [8]:
from abc import ABC, abstractmethod
from datetime import datetime
import time

class PaymentProcessor(ABC):

    @abstractmethod
    def process_payment(self, amount):
        pass


class Cash(PaymentProcessor):
    def process_payment(self, amount):
        print(f"[NAKİT] {float(amount)} TL nakit odeme alindi.")

class PayWithCreditCard(PaymentProcessor):
    def __init__(self, credit_card_number):
        self.credit_card_number = credit_card_number
        
    def process_payment(self, amount):
        print(f"[KREDİ KARTI] {self.credit_card_number} numarali karttan {amount} TL tahsil edildi.")

class BankTransfer(PaymentProcessor):
    def __init__(self, iban):
        self.iban = iban

    def process_payment(self, amount):
        print(f"[BANKA HAVALESİ] {self.iban} iban numarasina {amount} TL transfer onaylandi.")


class NotificationService(ABC):
    @abstractmethod
    
    def send_notification(self, passenger, message):
        pass

class Email(NotificationService):

    
    def send_notification(self, passenger, message):
        print(f"[EMAIL GONDERILDI] Alıcı: {passenger.email} --> Mesaj: {message}")

    
class SMS(NotificationService):
    
    def send_notification(self, passenger, message):
        print(f"[SMS GONDERILDI] Alıcı: {passenger.phone_no} --> Mesaj: {message}.")
        

class Logger():
    
    logs = []

    @staticmethod # staticmethod yazılmayınca description'u örnekleme yapmak için self gibi kullandı.
    def log(description):
        timestamp = datetime.now().strftime("%Y-%m-%d  %H.%M")
        
        record = {
            "timestamp": timestamp,
            "description": description
        }
        
        Logger.logs.append(record)
        print(f"[LOG {(timestamp)}] {description}")

    @classmethod
    def show_log_data(cls):
        if cls.logs:
            for log in cls.logs:
                print(log)

        else:
            print("kayit bulunamadi.")

    @staticmethod
    def log_transaction(func):
        def wrapper(*args, **kwargs):

            print(f"\n {func.__name__} islemi yapiliyor...\n")
            time.sleep(3)
            func(*args, **kwargs)
            print(f"\n {func.__name__} islemi basariyla tamamlandi.\n")
            timestamp = datetime.now().strftime("%Y-%m-%d  %H.%M")
            record = {
            "timestamp": timestamp,
            "description": f"{func.__name__}"}
            Logger.logs.append(record)
            
        return wrapper

   
            

        
        
class Flight():
    flight_data = []
    def __init__(self, flight_code, from_where, to_where, take_off_time, arrival_time, price, available_seat = 150):
        self.flight_code = flight_code
        self.from_where = from_where
        self.to_where = to_where
        self.take_off_time = take_off_time
        self.arrival_time = arrival_time
        self.price = price
        self.available_seat = available_seat
        Flight.flight_data.append(self)

    
    @classmethod
    def show_flight_data(cls, obj=None):
        if obj in cls.flight_data:
            return obj
        else:
            for flight in cls.flight_data:
                print(flight)

    @classmethod
    def find_by_code(cls, input_code):
        for f in cls.flight_data:
            if f.flight_code == input_code:
                return f
        raise Exception("Ucus Kaydi Bulunamadi.")
            
    
    def __repr__(self):
        return f"Ucus Bilgisi: Ucus: {self.flight_code} | {self.from_where} --> {self.to_where} | Kalkış: {self.take_off_time} | Varış: {self.arrival_time} | Müsait Koltuk Sayisi: {self.available_seat}"

class Passenger():
    passenger_data = []
    _p_counter = 0
    
    def __init__(self, name, email, phone_no):
        
        Passenger._p_counter += 1
        self.passenger_id = str(f"P{Passenger._p_counter:04d}")
        self.name = name
        self.email = email
        self.phone_no = phone_no
        Passenger.passenger_data.append(self)

    @classmethod
    @Logger.log_transaction
    def create_new_passenger(cls, name, email, phone_no):
        new_passenger = Passenger(name, email, phone_no)
        print(f"Sayin {new_passenger.name}, {new_passenger.passenger_id} kodlu Yolcu ID'niz olusturulmustur.")
        return new_passenger
        

    def __repr__(self):
        return f"{self.passenger_id} | {self.name}"

    @classmethod
    def show_passenger_data(cls, obj=None):
        if obj in cls.passenger_data:
            print(obj)
        else:
            for passenger in cls.passenger_data:
                print(passenger)

    @classmethod
    def find_by_id(cls, input_id):
        for p in cls.passenger_data:
            if p.passenger_id == input_id:
                return p
           
        raise Exception("Yolcu Kaydı Bulunamadi.") # maindeki try-except blokları sayesinde buradaki hatanın programı patlatması önleniyor.
        

class Ticket(): # tek uçuş ve yolcudan nesnesi oluşabilir.
    _counter = 0 # bütün biletler için.
    
    def __init__(self, code, passenger, flight):
        self.flight = flight
        self.passenger = passenger

        Ticket._counter =+ 1


class Rezervation():
    
    rezervation_data = []
    _rez_counter = 0

    
    
    def __init__(self, passenger, flight):
        self.passenger = passenger
        self.flight = flight
        Rezervation._rez_counter += 1 # direkt sınıfa ulaşmak için.
        self.rez_id = str(f"B{Rezervation._rez_counter:04d}")
        self.ticket = None
        self.is_paid = False
        self.status = "Beklemede"
        Rezervation.rezervation_data.append(self)
    
    @staticmethod
    @Logger.log_transaction
    def booking(passenger, flight):
        rez = Rezervation(passenger, flight)
        Logger().log(f"Sayin {rez.passenger.name}, {rez.rez_id} numarali rezervasyonunuz olusturulmuştur. Rezervasyonunuzun onaylanması için odeme yapınız.\n")
        return rez
            

    def calculate_total_price(self):
        total_price = self.flight.price
        return total_price
            
    @Logger.log_transaction
    def checkout(self, payment_method: PaymentProcessor): # parantez içindeki ifade türeme için zorunlu değil ama kodu okuyanlar için bilgilendirici.
        
        if self.is_paid == True:
            raise Exception(f"Odeme zaten alindi.")
            
        
        final_amount = self.calculate_total_price()
        payment_method.process_payment(final_amount)
        
        time.sleep(2)
        
        try:
            self.is_paid = True
            if self.is_paid and self.flight.available_seat > 0:
                self.status = "Onaylandi"
                self.ticket = Ticket(self.rez_id, self.passenger, self.flight)
                self.flight.available_seat -= 1
                Logger().log(f"(Booking {self.rez_id}) --> Rezervasyon {self.rez_id} onaylandi.")
                Email().send_notification(self.passenger, f"Sayin {self.passenger.name}, {self.flight.flight_code} ucusunuz onaylanmiştir.")
                SMS().send_notification(self.passenger, f"Sayin {self.passenger.name}, {self.flight.flight_code} ucusunuz onaylanmıştir.")
                print(f"Rezervasyon ID: {self.rez_id} | Durum: {self.status}")
            
            
            

        except Exception as e:
            
            self.status = "Reddedildi."
            print(f"Rezervasyonunuz reddedildi. Hata Kodu: {e}")
            self.ticket = None
            print(f"Rezervasyon ID: {self.rez_id} | Durum: {self.status}")

    
            
         
    @Logger.log_transaction
    def cancel(self):
        print("[IPTAL SURECI]")
        self.status = "İptal edildi"
        Logger().log(f"(Booking {self.rez_id}) --> Rezervasyon {self.rez_id} iptal edildi.")
        print("Paranız iade ediliyor...")
        time.sleep(2)
        self.flight.available_seat += 1
        self.ticket = None
        Email().send_notification(self.passenger, f"Sayin {self.passenger.name}, {self.flight.flight_code} ucusunuz iptal edilmiştir.")
        SMS().send_notification(self.passenger, f"Sayin {self.passenger.name} {self.flight.flight_code} ucusunuz iptal edilmiştir.")
        print(f"Rezervasyon ID: {self.rez_id} | Durum: {self.status}")

    @classmethod
    def find_by_rez_id(cls, rez_id):
        for r in cls.rezervation_data:
            if str(r.rez_id) == rez_id:
                return r
            
        raise Exception("Rezervasyon Bulunamadi.")

    @classmethod
    @Logger.log_transaction # classmethod decorator'ünün altında.
    def show_rezervation_data(cls, obj=None): # nesne gönderildiyse spesifik olarak onun bilgilerinin yazılmasını istiyorum.
        if obj in cls.rezervation_data:
            print(obj)
        else:
            for rez in cls.rezervation_data:
                print(rez)
    
    

    

    def __str__(self):
        return f"\n|Rezervasyon ID: {self.rez_id} \n|Yolcu: {self.passenger} \n|Ucus Bilgileri: [{self.flight.from_where} --> {self.flight.to_where}]  [{self.flight.take_off_time} --> {self.flight.arrival_time}]  \n| Durum: {self.status}"
        
        

        



In [9]:

# ------------Örnek veriler------------- (ucus verilerinin tanımlanması için bu hücre çalıştırılmalı. Yolcu verilerinin tanımlanması tercihendir.)

mert_yilmaz = Passenger("Mert YILMAZ", "ymert9712@gmail.com", "05427911549")
tarik_furkan_atac = Passenger("Tarık Furkan Ataç", "tarikfatac@outlook.com", "05407841469")
gozde_idil_ozcan = Passenger("Gözde İdil Özcan", "idilgözcan@hotmail.com", "05354826597")
ege_bayraktar = Passenger("Ege Bayraktar", "begeraktar@gmail.com", "05321465795")

ucus1 = Flight("IB0626", "ISTANBUL", "BALIKESIR", "2026-05-12 -- 12.00", "2026-05-12 -- 13.00", 1000)
ucus2 = Flight("IA0689", "ISTANBUL", "ANKARA", "2026-06-06 -- 11.00", "2026-06-06 -- 12.45", 2500)
ucus3 = Flight("IK0642", "ISTANBUL", "KOCAELI", "2026-06-01 -- 19.50", "2026-06-01 -- 20.00", 700)
ucus4 = Flight("IA0062", "ISTANBUL", "ANTALYA", "2026-08-13 -- 06.50", "2026-08-13 -- 08.05", 4000)

# --------------------------------------

In [10]:
if __name__ == "__main__":
    while True:
        print(3*"\n")
        print("----HavaYolu Rezervasyon Sistemi----")
        print(3*"\n")
        print("1 - Yeni Yolcu Kaydi")
        print("2 - Ucuslari Listele")
        print("3 - Rezervasyon Yap")
        print("4 - Rezervasyon Bilgilerini Goruntule")
        print("5 - Rezervasyonu Iptal Et")
        print("6 - Odeme Yap & Bilet Kes")
        print("7 - Sistem Loglarini Goruntule")
        print("0 - Cikis")

        num = input("Seciminizi yapin: (0-5)")
        print(3*"\n")

       

        if num == "0":
            print("Sistemden Çıkılıyor.")
            break

        if num == "1":
            try:
                created_passenger = Passenger.create_new_passenger(input("\nİsim - Soyisim giriniz: \n"), input("\nEposta Hesabınızı Giriniz: \n"), input("\nTelefon Numaranızı Giriniz: \n"))
               
                
            except Exception as e:
                print(f"Yolcu kaydi olusturalamadi. Hata kodu: {e}")

            
               
            
            
        
        if num == "2":
            try: 
                Flight.show_flight_data()
                
            except Exception as e:
                print(f"Ucus verilerini ulasilamadi. Hata Kodu: {e}")

        

        if num == "3":
            try:
                rez = Rezervation.booking(Passenger.find_by_id(input("rezervasyon yapılacak kişinin Yolcu ID'sini giriniz: \n")), Flight.find_by_code(input("Ucus kodunu giriniz: \n")))

            except Exception as e:
                print(f"Rezervasyon yapilamadi. Hata Kodu: {e}")


        if num == "4":
            try:
                rez_info = Rezervation.show_rezervation_data(Rezervation.find_by_rez_id(input("Lutfen bilgilerini goruntulemek istediginiz planın Rezervasyon ID'sini giriniz: ")))
            except Exception as e:
                print(f"Rezervasyon bilgileri bulunamadi. Hata Kodu: {e}")
                

        
        if num == "5":
            try:
                iptal_rez = Rezervation.find_by_rez_id(input("İptal etmek istediğiniz planın Rezervasyon ID'sini girin: \n"))
                iptal_rez.cancel()

            except Exception as e:
                print(f"Rezervasyon iptal edilemedi. Hata Kodu: {e}")
                
                
                

        if num == "6":
            try:
                
                rez = Rezervation.find_by_rez_id(input("Ödemesi yapılacak planın Rezervasyon ID'sini girin: \n"))
                pay_with = input("\nÖdeme yolunu seçiniz: \n(1 - Nakit \n2 - Kredi Kartı \n3 - Banka Havalesi)\n")
                if pay_with == "1":
                    nakit_odeme = Cash()
                    rez.checkout(nakit_odeme)
                elif pay_with == "2":
                    kredi_karti = PayWithCreditCard(input("Kredi Kartı Numarası Giriniz: "))
                    rez.checkout(kredi_karti)
                elif pay_with == "3":
                    havale = BankTransfer(input("Havale yapılacak IBAN Giriniz: "))
                    rez.checkout(havale)

            except Exception as e:
                print(f"Ödeme Gerçekleşemedi. Hata Kodu: {e}")

        if num == "7":
            try:
                Logger.show_log_data()

            except Exception as e:
                print(f"Kayıtlar yüklenemedi. Hata Kodu: {e}")

        





----HavaYolu Rezervasyon Sistemi----




1 - Yeni Yolcu Kaydi
2 - Ucuslari Listele
3 - Rezervasyon Yap
4 - Rezervasyon Bilgilerini Goruntule
5 - Rezervasyonu Iptal Et
6 - Odeme Yap & Bilet Kes
7 - Sistem Loglarini Goruntule
0 - Cikis


Seciminizi yapin: (0-5) 0






Sistemden Çıkılıyor.
